In [1]:
# ================================================================
#  LCCN-R13 - PER-SEED PAIRED STATISTICS ON STORED PREDICTIONS  [CPU]
#  Answers IJIES Reviewer 1, point 3 (mixed inferential levels).
#  For every claim comparison and EVERY seed whose per-comment
#  predictions exist, on the same 6,000 test comments:
#    paired bootstrap (B = 10,000) of the macro-F1 difference,
#      95% percentile CI and two-sided p
#    McNemar's test with continuity correction on per-comment
#      correctness, with discordant counts b and c
#  p-values are Holm-corrected ACROSS SEEDS within each comparison.
#  The between-seed level (mean, std of the per-seed macro-F1) is
#  reported separately and never mixed with the within-seed tests.
#
#  Seed provenance of prediction files (trainer v1.4 overwrites one
#  file per model x arm with the LAST seed run):
#    paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__{arm}.csv   -> seed 42
#    paper1-nb6-lccn-xlm43-44/preds/XLM-RoBERTa__{arm}.csv     -> seed 44
#    any  .../preds/XLM-RoBERTa__{arm}__seed{N}.csv            -> seed N
#      (produced by the patched trainer in the later GPU notebooks)
#  Add Input: paper1-nb4-lccn-xlmroberta (output),
#             paper1-nb6-lccn-xlm43-44 (output), and later the
#             notebook output that holds the __seed files.
#  Accelerator None. Save & Run All (Commit). Transcribe ONLY from log.
# ================================================================
import glob, os, re, sys
import numpy as np, pandas as pd
from scipy.stats import chi2
from sklearn.metrics import f1_score
print("python", sys.version.split()[0])

MODEL = "XLM-RoBERTa"
COMPARISONS = [("raw", "indicsoundex"), ("raw", "lccn"), ("raw", "lccn_base"),
               ("raw", "lccn_indicxlit"), ("lccn_base", "lccn")]
B = 10000
RNG_SEED = 0

def seed_of(path):
    m = re.search(r"__seed(\d+)\.csv$", path)
    if m: return int(m.group(1))
    if "nb4-lccn-xlmroberta" in path: return 42
    if "nb6-lccn-xlm43-44" in path: return 44
    return None

files = sorted(glob.glob(f"/kaggle/input/**/preds/{MODEL}__*.csv", recursive=True))
print("prediction files found:")
preds = {}   # (arm, seed) -> path
for f in files:
    arm = re.sub(r"__seed\d+$", "", os.path.basename(f)[len(MODEL)+2:-4])
    s = seed_of(f)
    print(f"  seed {s}  {arm:<18} {f}")
    if s is None: continue
    preds[(arm, s)] = f
print(f"\n{len(preds)} (arm, seed) prediction sets usable")


python 3.12.13
prediction files found:
  seed 42  indicsoundex       /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__indicsoundex.csv
  seed 42  indicsoundex_guard /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__indicsoundex_guard.csv
  seed 42  keel               /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__keel.csv
  seed 42  kope               /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__kope.csv
  seed 42  lccn               /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__lccn.csv
  seed 42  lccn_base          /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__lccn_base.csv
  seed 42  lccn_indicxlit     /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__lccn_indicxlit.csv
  seed 42  lccn_itrans        /kaggle/input/notebooks/tejaswinina/paper1-nb4-

In [2]:
# ================================================================
#  CELL 1 - LOAD, ALIGN, TESTS
# ================================================================
def load(path):
    d = pd.read_csv(path, dtype=str, keep_default_na=False)
    d.columns = [c.strip().lower() for c in d.columns]
    idc  = next(c for c in d.columns if c in ("id", "comment_id"))
    true = next(c for c in d.columns if "true" in c or c == "gold")
    pred = next(c for c in d.columns if "pred" in c)
    d = d[[idc, true, pred]].copy(); d.columns = ["id", "y", "p"]
    for c in ("y", "p"): d[c] = d[c].str.strip().str.lower()
    return d.set_index("id")

def macro_f1(y, p): return f1_score(y, p, average="macro")

def paired_bootstrap(y, pa, pb, B=B, seed=RNG_SEED):
    rng = np.random.default_rng(seed); n = len(y)
    obs = macro_f1(y, pb) - macro_f1(y, pa)
    deltas = np.empty(B)
    for i in range(B):
        idx = rng.integers(0, n, n)
        deltas[i] = macro_f1(y[idx], pb[idx]) - macro_f1(y[idx], pa[idx])
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    p = 2 * min((deltas <= 0).mean(), (deltas >= 0).mean()); p = min(p, 1.0)
    return obs, lo, hi, p

def mcnemar(y, pa, pb):
    oka, okb = (pa == y), (pb == y)
    b = int((oka & ~okb).sum()); c = int((~oka & okb).sum())
    stat = (abs(b - c) - 1) ** 2 / (b + c) if (b + c) else 0.0
    return b, c, stat, float(chi2.sf(stat, 1))

def holm(ps):
    ps = np.asarray(ps, float); m = len(ps); order = np.argsort(ps)
    adj = np.empty(m); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * ps[i]); adj[i] = min(running, 1.0)
    return adj

rows = []
for a, b_ in COMPARISONS:
    seeds = sorted(s for (arm, s) in preds if arm == a and (b_, s) in preds)
    if not seeds:
        print(f"\n{a} vs {b_}: no seed with both arms stored (pending GPU reruns)"); continue
    print(f"\n{a} vs {b_}: seeds {seeds}")
    for s in seeds:
        A, Bd = load(preds[(a, s)]), load(preds[(b_, s)])
        j = A.join(Bd, lsuffix="_a", rsuffix="_b", how="inner")
        assert len(j) == 6000, f"alignment gave {len(j)} rows"
        assert (j.y_a == j.y_b).all(), "gold labels disagree"
        y, pa, pb = j.y_a.values, j.p_a.values, j.p_b.values
        fa, fb = macro_f1(y, pa), macro_f1(y, pb)
        obs, lo, hi, pboot = paired_bootstrap(y, pa, pb)
        bb, cc, stat, pmc = mcnemar(y, pa, pb)
        rows.append(dict(comparison=f"{a} vs {b_}", seed=s, f1_a=round(fa, 4), f1_b=round(fb, 4),
                         delta=round(obs, 4), ci_low=round(lo, 4), ci_high=round(hi, 4),
                         p_boot=round(pboot, 4), b_disc=bb, c_disc=cc,
                         chi2=round(stat, 2), p_mcnemar=round(pmc, 5)))
        print(f"  seed {s}: {a} {fa:.4f}  {b_} {fb:.4f}  delta {obs:+.4f} "
              f"[{lo:+.4f}, {hi:+.4f}] p_boot {pboot:.4f} | McNemar b {bb} c {cc} "
              f"chi2 {stat:.2f} p {pmc:.5f}")

tab = pd.DataFrame(rows)
for comp, g in tab.groupby("comparison"):
    tab.loc[g.index, "p_boot_holm"]    = holm(g.p_boot.values).round(4)
    tab.loc[g.index, "p_mcnemar_holm"] = holm(g.p_mcnemar.values).round(5)
tab.to_csv("/kaggle/working/lccn_r13_perseed_stats.csv", index=False)
print("\n=== WITHIN-SEED PAIRED TESTS (Table 8 rows) ===")
print(tab.to_string(index=False))



raw vs indicsoundex: seeds [42, 44]
  seed 42: raw 0.8058  indicsoundex 0.7773  delta -0.0285 [-0.0380, -0.0190] p_boot 0.0000 | McNemar b 520 c 339 chi2 37.72 p 0.00000
  seed 44: raw 0.8032  indicsoundex 0.7839  delta -0.0193 [-0.0289, -0.0099] p_boot 0.0000 | McNemar b 478 c 370 chi2 13.50 p 0.00024

raw vs lccn: seeds [42, 44]
  seed 42: raw 0.8058  lccn 0.7991  delta -0.0067 [-0.0152, +0.0022] p_boot 0.1476 | McNemar b 392 c 351 chi2 2.15 p 0.14225
  seed 44: raw 0.8032  lccn 0.8066  delta +0.0034 [-0.0041, +0.0110] p_boot 0.3830 | McNemar b 264 c 286 chi2 0.80 p 0.37055

raw vs lccn_base: seeds [42]
  seed 42: raw 0.8058  lccn_base 0.7991  delta -0.0067 [-0.0150, +0.0017] p_boot 0.1276 | McNemar b 361 c 312 chi2 3.42 p 0.06428

raw vs lccn_indicxlit: seeds [42]
  seed 42: raw 0.8058  lccn_indicxlit 0.8130  delta +0.0072 [-0.0012, +0.0156] p_boot 0.0886 | McNemar b 307 c 355 chi2 3.34 p 0.06774

lccn_base vs lccn: seeds [42]
  seed 42: lccn_base 0.7991  lccn 0.7991  delta +0.0000

In [3]:
# ================================================================
#  CELL 2 - BETWEEN-SEED LEVEL AND VERDICT
#  Mean and std over the stored seeds, from the same prediction
#  files, kept separate from the within-seed tests above.
# ================================================================
print("=== BETWEEN-SEED (per-seed macro-F1 of each arm, from stored files) ===")
arms = sorted({arm for (arm, s) in preds})
summary = []
for arm in arms:
    f1s = {}
    for (a, s), f in preds.items():
        if a == arm:
            d = load(f); f1s[s] = macro_f1(d.y.values, d.p.values)
    vals = np.array(list(f1s.values()))
    summary.append(dict(arm=arm, seeds=sorted(f1s), n=len(vals),
                        mean=round(vals.mean(), 4), std=round(vals.std(ddof=0), 4) if len(vals) > 1 else None))
    print(f"  {arm:<18} seeds {sorted(f1s)}  " + "  ".join(f"{s}:{v:.4f}" for s, v in sorted(f1s.items()))
          + (f"  mean {vals.mean():.4f} std {vals.std(ddof=0):.4f}" if len(vals) > 1 else ""))
pd.DataFrame(summary).to_csv("/kaggle/working/lccn_r13_between_seed.csv", index=False)

print("\n=== VERDICT PER COMPARISON (within-seed, Holm-corrected McNemar) ===")
for comp, g in tab.groupby("comparison"):
    sig = g[g.p_mcnemar_holm < 0.05]
    sign = "worse" if g.delta.mean() < 0 else "better"
    print(f"  {comp:<24} seeds {list(g.seed)}  significant at {len(sig)}/{len(g)} seed(s)  "
          f"mean delta {g.delta.mean():+.4f}  ({sign} than first arm)")
missing = [(a, b_) for a, b_ in COMPARISONS
           if not any((a, s) in preds and (b_, s) in preds for s in (42, 43, 44))]
print("\nSTATUS")
print("  seeds with all three claim arms stored :", sorted({s for (arm, s) in preds if all((x, s) in preds for x in ('raw','indicsoundex','lccn'))}))
print("  comparisons with no stored seed        :", missing or "none")
print("  seed 43 and the lccn_base / lccn_indicxlit seeds 43-44 require the GPU rerun")
print("  with per-seed prediction files (__seed{N}.csv); rerun this notebook after it.")
print("\nDONE - lccn_r13_perseed_stats.csv and lccn_r13_between_seed.csv written.")
print("Save & Run All (Commit); transcribe only from the committed log.")


=== BETWEEN-SEED (per-seed macro-F1 of each arm, from stored files) ===
  indicsoundex       seeds [42, 44]  42:0.7773  44:0.7839  mean 0.7806 std 0.0033
  indicsoundex_guard seeds [42]  42:0.7861
  keel               seeds [42]  42:0.8004
  kope               seeds [42]  42:0.8090
  lccn               seeds [42, 44]  42:0.7991  44:0.8066  mean 0.8029 std 0.0037
  lccn_base          seeds [42]  42:0.7991
  lccn_indicxlit     seeds [42]  42:0.8130
  lccn_itrans        seeds [42]  42:0.8051
  metaphone          seeds [42]  42:0.7916
  raw                seeds [42, 44]  42:0.8058  44:0.8032  mean 0.8045 std 0.0013
  soundex            seeds [42]  42:0.7786
  soundex_guard      seeds [42]  42:0.7792
  terun              seeds [42]  42:0.8041

=== VERDICT PER COMPARISON (within-seed, Holm-corrected McNemar) ===
  lccn_base vs lccn        seeds [42]  significant at 0/1 seed(s)  mean delta +0.0000  (better than first arm)
  raw vs indicsoundex      seeds [42, 44]  significant at 2/2 seed(s)  